## 1. Khởi tạo Môi trường và Cấu hình Tham số (Setup)

Bài toán này sử dụng các thư viện cần thiết và thiết lập các biến toàn cục cho mô hình:

1.  **Thư viện phân tích dữ liệu:**
    * `numpy`, `pandas`: Xử lý mảng và thao tác với dữ liệu bảng (DataFrame).
    * `matplotlib`, `seaborn`: Trực quan hóa dữ liệu và vẽ biểu đồ hiệu suất.

2.  **Thư viện Deep Learning (TensorFlow/Keras):**
    * **Layers:** Sử dụng các lớp `Conv2D`, `MaxPooling2D` để trích xuất đặc trưng và `Dense`, `Dropout` để phân loại.
    * **Regularization:** Sử dụng `l2` và `Dropout` để chống hiện tượng quá khớp (Overfitting).
    * **Callbacks:** `EarlyStopping` (dừng sớm), `ReduceLROnPlateau` (giảm tốc độ học) và `ModelCheckpoint` (lưu model tốt nhất).

3.  **Cấu hình Siêu tham số (Hyperparameters):**
    * `IMG_WIDTH`, `IMG_HEIGHT`: **224x224** (Kích thước chuẩn cho các mạng CNN hiện đại).
    * `BATCH_SIZE`: **32** (Số lượng ảnh xử lý trong một lần cập nhật trọng số).
    * `EPOCHS`: **30** (Số vòng lặp huấn luyện tối đa).
    * `LEARNING_RATE`: **0.0005** (Tốc độ học khởi tạo).

4.  **Thiết lập đường dẫn:** Code tự động kiểm tra môi trường chạy (Google Colab hoặc Local) để gắn (mount) Google Drive và trỏ đến thư mục dữ liệu chính xác.

In [ ]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow.keras.backend as K

IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 32
EPOCHS = 30
NUM_FOLDS = 5
LEARNING_RATE = 0.0005

try:
    from google.colab import drive
    print("Detected: GOOGLE COLAB environment")
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/AI_VGG16_Classifier/data/processed'
except ImportError:
    print("Detected: LOCAL environment")
    DATA_DIR = '../data/processed'

print(f"Data Directory: {DATA_DIR}")

2026-01-17 13:30:26.443761: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-17 13:30:29.585116: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-17 13:30:36.488082: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Detected: LOCAL environment
Đang tìm dữ liệu tại: ../data/raw


## 2. Tải Dữ liệu và Phân tích Trực quan (EDA)

Trước khi đưa vào huấn luyện, bước **Khám phá dữ liệu (Exploratory Data Analysis)** để đảm bảo chất lượng đầu vào:

1.  **Xây dựng DataFrame:** Hàm `load_image_paths` quét toàn bộ thư mục, trích xuất đường dẫn ảnh và nhãn lớp, sau đó xáo trộn ngẫu nhiên (Shuffle) để đảm bảo tính khách quan.
2.  **Kiểm tra Phân bố (Data Distribution):** Sử dụng biểu đồ `sns.countplot` để kiểm tra xem dữ liệu giữa các lớp có cân bằng không. Dữ liệu lệch (Imbalanced) có thể khiến mô hình học thiên vị.
3.  **Kiểm tra Chất lượng (Sanity Check):** Hiển thị ngẫu nhiên 9 tấm ảnh mẫu kèm nhãn thực tế để xác nhận lại quy trình tiền xử lý (cắt ảnh, gán nhãn) đã chính xác hay chưa.

In [ ]:
def load_image_paths(data_dir):
    image_dir = Path(data_dir)
    filepaths = list(image_dir.glob(r'**/*.jpg')) + list(image_dir.glob(r'**/*.png')) + list(image_dir.glob(r'**/*.jpeg'))
    labels = [os.path.split(os.path.split(filepath)[0])[1] for filepath in filepaths]

    filepaths = pd.Series(filepaths, name='Filepath').astype(str)
    labels = pd.Series(labels, name='Label')

    df = pd.concat([filepaths, labels], axis=1)
    df = df.sample(frac=1).reset_index(drop=True)
    return df

try:
    df = load_image_paths(DATA_DIR)
    print(f"Total images: {len(df)}")
    
    plt.figure(figsize=(10, 5))
    sns.countplot(x=df['Label'])
    plt.title("Data Distribution")
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.show()

    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        sample = df.sample(1).iloc[0]
        img = plt.imread(sample['Filepath'])
        ax.imshow(img)
        ax.set_title(sample['Label'])
        ax.axis('off')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Error: {e}")

Tổng số ảnh tìm thấy: 18

Số lượng ảnh mỗi lớp:
Label
pins_Ronaldo     6
pins_Benzenma    6
pins_Messi       6
Name: count, dtype: int64


## Định nghĩa Mô hình Custom CNN

Hàm `build_custom_cnn` khởi tạo một mạng nơ-ron tích chập với các đặc điểm kỹ thuật:

* **Input Shape:** `(224, 224, 3)`
* **Cấu trúc:** 4 khối Conv2D + BatchNormalization + MaxPooling.
* **Regularization:** Sử dụng kết hợp **L2 Regularization** ($0.001$) và **Dropout** (từ $0.2$ đến $0.5$) để giảm thiểu sai số kiểm tra.
* **Classifier:** Mạng Dense 512 units.
* **Optimizer:** Adam với Learning Rate được cấu hình sẵn.
* **Loss Function:** Categorical Crossentropy (Phân loại đa lớp).

In [ ]:
def build_custom_cnn(num_classes):
    model = Sequential()
    
    model.add(Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3)))
    
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.3))
    
    model.add(Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(0.001)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.4))
    
    model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.4))

    model.add(Flatten())
    
    model.add(Dense(512, activation='relu', kernel_regularizer=l2(0.01)))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    
    model.add(Dense(num_classes, activation='softmax'))
    
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

print("Custom CNN Model built successfully.")

Đã khởi tạo hàm build_custom_cnn thành công!


## 4. Tăng cường Dữ liệu (Data Augmentation) & Chuẩn hóa

Cấu hình chi tiết cho `ImageDataGenerator`:

1.  **Chuẩn hóa (Normalization):**
    * `rescale=1./255`: Chuyển đổi giá trị pixel từ [0, 255] về khoảng **[0, 1]**. Điều này giúp mô hình tính toán nhanh hơn và hội tụ tốt hơn.

2.  **Biến đổi hình học (Geometric Transformations):**
    * `rotation_range=30`: Xoay ảnh ngẫu nhiên 30 độ.
    * `width/height_shift`: Dịch chuyển ảnh sang trái/phải/lên/xuống.
    * `zoom_range`, `shear_range`: Phóng to và làm méo ảnh nhẹ để mô hình tập trung vào các chi tiết khuôn mặt thay vì khung cảnh.
    * `horizontal_flip`: Lật ảnh ngang (tăng gấp đôi lượng dữ liệu).

3.  **Biến đổi quang học (Photometric Transformations):**
    * `brightness_range=[0.7, 1.3]`: Thay đổi độ sáng ngẫu nhiên (tối đi 30% hoặc sáng lên 30%). **Mục đích:** Giúp mô hình nhận diện tốt khuôn mặt trong các điều kiện ánh sáng khác nhau, tránh phụ thuộc vào màu sắc môi trường.

> Tập Validation (`val_datagen`) chỉ được chuẩn hóa (`rescale`), **không** áp dụng các biến đổi ngẫu nhiên để đảm bảo việc đánh giá là khách quan.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)


Training Custom CNN for Fold 1 ...
Found 14 validated image filenames belonging to 3 classes.
Found 4 validated image filenames belonging to 3 classes.


2026-01-17 13:30:42.290273: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-01-17 13:30:42.531614: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 51380224 exceeds 10% of free system memory.
2026-01-17 13:30:42.833745: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 51380224 exceeds 10% of free system memory.
2026-01-17 13:30:42.932960: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 51380224 exceeds 10% of free system memory.


Epoch 1/20


2026-01-17 13:30:44.707018: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 51380224 exceeds 10% of free system memory.
2026-01-17 13:30:46.542761: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 89915392 exceeds 10% of free system memory.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.3571 - loss: 1.1004
Epoch 1: val_accuracy improved from None to 0.00000, saving model to ../models/custom_cnn_fold_1.h5


...


Epoch 2: finished saving model to ../models/custom_cnn_fold_3.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - accuracy: 0.4286 - loss: 2.8779 - val_accuracy: 0.5000 - val_loss: 2.5077
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.4286 - loss: 2.6209
Epoch 3: val_accuracy did not improve from 0.50000
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.4286 - loss: 2.6209 - val_accuracy: 0.5000 - val_loss: 1.1843
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3571 - loss: 1.5178
Epoch 4: val_accuracy did not improve from 0.50000
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3571 - loss: 1.5178 - val_accuracy: 0.2500 - val_loss: 1.3263
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3571 - loss: 1.1438
Epoch 5: val_accuracy did not improve from 0.50000
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.3571 - loss: 1.1438 - val_accuracy: 0.2500 - val_loss: 1.2701
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5714 - loss: 1.0496
Epoch 6: 


Epoch 1: finished saving model to ../models/custom_cnn_fold_4.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.2667 - loss: 1.0732 - val_accuracy: 0.0000e+00 - val_loss: 4.5807
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.3333 - loss: 2.8211
Epoch 2: val_accuracy improved from 0.00000 to 0.66667, saving model to ../models/custom_cnn_fold_4.h5



Epoch 2: finished saving model to ../models/custom_cnn_fold_4.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - accuracy: 0.3333 - loss: 2.8211 - val_accuracy: 0.6667 - val_loss: 2.3731
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.2000 - loss: 4.2338
Epoch 3: val_accuracy did not improve from 0.66667
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.2000 - loss: 4.2338 - val_accuracy: 0.0000e+00 - val_loss: 2.0141
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3333 - loss: 1.7846
Epoch 4: val_accuracy did not improve from 0.66667
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.3333 - loss: 1.7846 - val_accuracy: 0.0000e+00 - val_loss: 1.4051
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.3333 - loss: 1.1823
Epoch 5: val_accuracy did not improve from 0.66667
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.3333 - loss: 1.1823 - val_accuracy: 0.0000e+00 - val_loss: 1.2765
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.3333 - loss: 1


Epoch 1: finished saving model to ../models/custom_cnn_fold_5.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - accuracy: 0.2667 - loss: 1.1088 - val_accuracy: 0.0000e+00 - val_loss: 1.9602
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.4667 - loss: 1.5769
Epoch 2: val_accuracy improved from 0.00000 to 0.33333, saving model to ../models/custom_cnn_fold_5.h5



Epoch 2: finished saving model to ../models/custom_cnn_fold_5.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.4667 - loss: 1.5769 - val_accuracy: 0.3333 - val_loss: 1.2011
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.1333 - loss: 2.0463
Epoch 3: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.1333 - loss: 2.0463 - val_accuracy: 0.3333 - val_loss: 1.1445
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.5333 - loss: 1.0070
Epoch 4: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - accuracy: 0.5333 - loss: 1.0070 - val_accuracy: 0.0000e+00 - val_loss: 1.3689
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.4000 - loss: 1.0586
Epoch 5: val_accuracy did not improve from 0.33333
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.4000 - loss: 1.0586 - val_accuracy: 0.3333 - val_loss: 1.5324
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5333 - loss: 0.9488
E

## 5. Huấn luyện và Đánh giá Mô hình (K-Fold Cross-Validation)

Để đảm bảo mô hình có độ tin cậy cao và không bị phụ thuộc vào một cách chia dữ liệu cụ thể, nhóm sử dụng kỹ thuật **Kiểm chứng chéo K-Fold (5-Fold Cross-Validation)**. 

Quy trình thực hiện trong mỗi vòng lặp (Fold) như sau:

1.  **Phân chia dữ liệu (Data Splitting):** Tại mỗi fold, dữ liệu được chia thành 2 phần: **Train (80%)** để học và **Validation (20%)** để kiểm tra, đảm bảo mọi tấm ảnh đều được dùng để kiểm tra ít nhất một lần.
2.  **Khởi tạo Generator:** Thiết lập luồng dữ liệu riêng biệt cho tập Train và Val của fold hiện tại.
3.  **Xây dựng Mô hình (Model Initialization):** Quan trọng nhất, ta gọi lại hàm `build_custom_cnn` để khởi tạo một mô hình hoàn toàn mới (reset trọng số), đảm bảo kiến thức của fold trước không ảnh hưởng đến fold sau.
4.  **Cấu hình Callbacks (Cơ chế giám sát):**
    * `ModelCheckpoint`: Chỉ lưu lại phiên bản mô hình có độ chính xác cao nhất (Best Weights).
    * `EarlyStopping`: Tự động dừng huấn luyện nếu mô hình không cải thiện sau 7 epochs (tránh lãng phí thời gian và Overfitting).
    * `ReduceLROnPlateau`: Giảm tốc độ học nếu Loss bị chững lại, giúp mô hình tìm được điểm tối ưu toàn cục (Global Minima).
5.  **Quản lý tài nguyên:** Sử dụng `K.clear_session()` sau mỗi fold để giải phóng bộ nhớ GPU/RAM.

In [ ]:
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
histories = []
acc_per_fold = []
loss_per_fold = []
fold_no = 1

for train_index, val_index in kf.split(df):
    print(f"\nTRAINING FOLD {fold_no}/{NUM_FOLDS}")
    
    train_data = df.iloc[train_index]
    val_data = df.iloc[val_index]
    
    train_gen = train_datagen.flow_from_dataframe(
        train_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    
    val_gen = val_datagen.flow_from_dataframe(
        val_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    
    model = build_custom_cnn(len(train_gen.class_indices))
    
    checkpoint_path = f"../models/custom_cnn_fold_{fold_no}.h5"
    if 'google.colab' in str(get_ipython()):
         checkpoint_path = f"/content/drive/MyDrive/models/custom_cnn_fold_{fold_no}.h5"

    callbacks = [
        ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
        EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001, verbose=1)
    ]
    
    try:
        history = model.fit(
            train_gen,
            epochs=EPOCHS, 
            validation_data=val_gen,
            callbacks=callbacks,
            verbose=1
        )
        
        histories.append(history)
        scores = model.evaluate(val_gen, verbose=0)
        print(f'Fold {fold_no} Accuracy: {scores[1]*100:.2f}%')
        acc_per_fold.append(scores[1] * 100)
        loss_per_fold.append(scores[0])
        
    except Exception as e:
        print(f"Error in Fold {fold_no}: {e}")

    K.clear_session()
    fold_no += 1

print(f"\nAVERAGE ACCURACY: {np.mean(acc_per_fold):.2f}%")

## 6. Trực quan hóa Kết quả Huấn luyện (Learning Curves)

Biểu đồ để đánh giá mô hình:

1.  **Biểu đồ Accuracy (Độ chính xác):**
    * Đường **Training Acc** (xanh) và **Validation Acc** (cam) nên tăng dần và tiệm cận nhau.
    * Nếu Training tăng cao nhưng Validation thấp lẹt đẹt $\rightarrow$ **Overfitting** (Học vẹt).
    * Nếu cả hai đều thấp $\rightarrow$ **Underfitting** (Chưa học được gì).

2.  **Biểu đồ Loss (Hàm mất mát):**
    * Cả hai đường nên giảm dần theo thời gian (Epochs).
    * Validation Loss là chỉ số quan trọng nhất để quyết định điểm dừng (Early Stopping).

> **Mục tiêu:** Nhóm mong muốn đường Validation bám sát đường Training, chứng tỏ mô hình có khả năng tổng quát hóa tốt trên dữ liệu lạ.

In [ ]:
def plot_training_history(histories):
    for i, history in enumerate(histories):
        acc = history.history['accuracy']
        val_acc = history.history['val_accuracy']
        loss = history.history['loss']
        val_loss = history.history['val_loss']
        epochs = range(len(acc))

        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.plot(epochs, acc, label='Training Acc')
        plt.plot(epochs, val_acc, label='Validation Acc')
        plt.title(f'Fold {i+1}: Accuracy')
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(epochs, loss, label='Training Loss')
        plt.plot(epochs, val_loss, label='Validation Loss')
        plt.title(f'Fold {i+1}: Loss')
        plt.legend()
        
        plt.show()

plot_training_history(histories)

### Bước 7: Phân tích Hiệu năng Mô hình

Đánh giá hiệu năng mô hình thông qua các bảng số liệu sau:

| Công cụ | Mục đích phân tích |
| :--- | :--- |
| **Confusion Matrix** | Cho biết mô hình hay bị nhầm lẫn cặp nào nhất (Ví dụ: Nhầm Messi thành Benzema). |
| **Precision** | Đo lường độ "tự tin" đúng của mô hình. |
| **Recall** | Đo lường khả năng "bao phủ" (không bỏ sót) của mô hình. |
| **F1-Score** | Chỉ số tổng hợp, F1 càng gần 1.0 (100%) thì mô hình càng hoàn hảo. |

In [ ]:
best_model_path = "/content/drive/MyDrive/models/custom_cnn_fold_1.h5"

if os.path.exists(best_model_path):
    print(f"Evaluating Best Model: {best_model_path}")
    loaded_model = load_model(best_model_path)
    
    test_gen = val_datagen.flow_from_dataframe(
        df, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    
    Y_pred = loaded_model.predict(test_gen)
    y_pred = np.argmax(Y_pred, axis=1)
    y_true = test_gen.classes
    class_labels = list(test_gen.class_indices.keys())
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_labels))

else:
    print("Model file not found.")